[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rudrite/kernels/blob/main/labs/stage-1/lab-1.1-first-kernels.ipynb)

# LAB·1.1 · First kernels: add, transpose

**Hardware:** any machine. `interpret=True` runs the kernel logic in Python; on a TPU runtime the same code compiles for real.

A Pallas kernel is a function over *refs*: windows into arrays that the runtime has already staged into fast memory. You write what one grid step does to its window; BlockSpecs and the grid decide which window that is. This lab builds that mental model with the two smallest possible kernels.

In [ ]:
import time
import numpy as np
import jax
import jax.numpy as jnp
from jax.experimental import pallas as pl

print(jax.__version__, jax.devices())
ON_TPU = jax.devices()[0].platform == "tpu"
INTERP = not ON_TPU  # interpret mode anywhere; compiled kernels on a real TPU

def check(name, got, want, tol=2e-2):
    err = float(jnp.abs(got.astype(jnp.float32) - want.astype(jnp.float32)).max())
    status = "ok" if err <= tol else "FAIL"
    print(f"{name}: max err {err:.3e} [{status}]")
    assert err <= tol, name


## The smallest kernel

No grid, no blocking: the whole array is one window. The kernel body is the algorithm; there is no schedule yet to speak of.

In [ ]:
def add_kernel(x_ref, y_ref, o_ref):
    o_ref[...] = x_ref[...] + y_ref[...]

def add(x, y):
    return pl.pallas_call(
        add_kernel,
        out_shape=jax.ShapeDtypeStruct(x.shape, x.dtype),
        interpret=INTERP,
    )(x, y)

x = jax.random.normal(jax.random.key(0), (256, 256), jnp.float32)
y = jax.random.normal(jax.random.key(1), (256, 256), jnp.float32)
check("add", add(x, y), x + y)

## Transpose: your first schedule decision

Same trivial body, but now the *index maps* do the work: the input block at grid position (i, j) writes to output position (j, i). The algorithm is `swap axes`; where each tile comes from and goes to is pure schedule.

In [ ]:
def transpose_kernel(x_ref, o_ref):
    o_ref[...] = x_ref[...].T

def transpose(x, bm=128, bn=128):
    m, n = x.shape
    return pl.pallas_call(
        transpose_kernel,
        grid=(m // bm, n // bn),
        in_specs=[pl.BlockSpec((bm, bn), lambda i, j: (i, j))],
        out_specs=pl.BlockSpec((bn, bm), lambda i, j: (j, i)),
        out_shape=jax.ShapeDtypeStruct((n, m), x.dtype),
        interpret=INTERP,
    )(x)

check("transpose", transpose(x), x.T)

## Exercise

Write `scale_bias(x, s, b)` computing `x * s + b` where `s` and `b` are scalars broadcast over a blocked 2D array. Two things to decide: how scalars reach the kernel (hint: close over them, or pass (1, 1) blocks), and what the grid is. Verify with `check` before moving on.